# Algoritmos de optimización - Seminario<br>
Nombre y Apellidos: Jorge Nozal Martin <br>
Url: https://github.com/.../03MAIR---Algoritmos-de-Optimizacion---2019/tree/master/SEMINARIO<br>

 
Problema:
> 1. Sesiones de doblaje <br>

**Descripción del problema:**

Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de grabación independientemente del número de tomas que se graben. No es posible grabar más de 6 tomas por día. 

El objetivo es planificar las sesiones por día de manera que el gasto por los servicios de los actores de doblaje sea el menor posible. Los datos son:

Número de actores: 10

Número de tomas : 30

Actores/Tomas : https://bit.ly/36D8IuK


(*) La respuesta es obligatoria
                                        

In [2]:
!pip install sympy

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


In [3]:
# Imports
import math
import itertools
import pandas as pd
import random
import numpy as np
from sympy import bell



**1.(*)¿Cuantas posibilidades hay sin tener en cuenta las restricciones?<br>**


El número de formas de agrupar un conjunto de n elementos en subconjuntos no vacíos se corresponde con el número de Bell.
En nuestro caso, tenemos 30 tomas que podrían organizarse en diferentes días de grabación, sin límite máximo de tomas por día. Por tanto, el número de posibilidades viene dado por:

B(30)

Este valor es extremadamente grande, lo que muestra que la enumeración exhaustiva del problema completo no es factible

In [4]:
# Cálculo del número de Bell para 30 tomas
n = 30  # número de tomas
numero_bell = bell(n)
print(f"Número de particiones sin restricciones (número de Bell) B({n}) = {numero_bell}")



Número de particiones sin restricciones (número de Bell) B(30) = 846749014511809332450147


**2.¿Cuantas posibilidades hay teniendo en cuenta todas las restricciones.**

Si ahora imponemos que no pueden grabarse más de 6 tomas por día, debemos contar las formas de particionar 30 tomas en grupos de tamaño entre 1 y 6.
Esto se puede resolver mediante backtracking, que evita contar permutaciones equivalentes. El resultado es todavía muy grande, pero menor que en el caso anterior.

Este análisis confirma que incluso con restricciones, el espacio de búsqueda es combinatoriamente explosivo, y la fuerza bruta no es viable.

In [5]:
# Ahora consideramos que no se pueden grabar más de 6 tomas por día.  
# Usamos un **backtracking** para contar las formas válidas de agrupar 30 tomas en bloques de 1 a 6.

def contar_particiones(n_tomas, max_tomas_por_dia, actual=[]):
    if n_tomas == 0:            #No grabar tomas en un día es posible asi que cuenta 1
        return 1
    
    if n_tomas < 0:             #Si nos pasamos de n_tomas ya no sigue contando participaciones
        return 0

    total = 0                   # Conteo de las particiones válidas

    # Para evitar permutaciones iguales, aseguramos que los bloques estén ordenados de menor a mator 
    start = actual[-1] if actual else 1

    for i in range(start, max_tomas_por_dia + 1):

        total += contar_particiones(n_tomas - i, max_tomas_por_dia, actual + [i])
        
    return total

total_particiones = contar_particiones(30, 6)
print(f"Número de particiones sin repetir orden y con bloques de ≤ 6: {total_particiones}")

Número de particiones sin repetir orden y con bloques de ≤ 6: 1206


Modelo para el espacio de soluciones<br>
**3.(*) ¿Cual es la estructura de datos que mejor se adapta al problema? Argumentalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, arguentalo)**


Respuesta

Al inicio partimos de un fichero CSV con la relación de tomas y actores. El primer paso fue transformarlo en una estructura manejable dentro de Python:

1. **Carga de datos con Pandas**

   * Leemos el archivo y limpiamos filas/columnas que no correspondían a tomas reales.
   * Cada columna corresponde a un actor, y cada fila a una toma.
   * Un valor 1.0 indica que el actor participa en esa toma.
   * Con esto obtenemos un DataFrame que nos permite explorar rápidamente el dataset.

2. Creación del **Diccionario de tomas**

   * A partir del DataFrame construimos un diccionario tomas: dict[int, set[int]].
   * La clave es el identificador de la toma, y el valor es un conjunto con los actores participantes.
   * Usar set evita duplicados y facilita comprobar si dos tomas comparten actores.

3. Una vez definido el diccionario de tomas, para implementar los distintos algoritmos necesitamos **dos estructuras adicionales**:

   * **Planificación**: list[list[int]]
      - Representa el reparto de tomas en días.
      - Cada lista interna es un día, con las tomas asignadas a ese día.
      - Esto es fundamental para calcular costes o generar vecinos en heurísticas.

   *  **Asistencia**: dict[int, set[int]]
      - Se utiliza al calcular el coste: indica los días en que cada actor debe asistir al estudio.
      - Así podemos contar cuántos días cobra cada actor y sumar el total.

#### Resumen

* **DataFrame inicial** → lectura de datos.
* **Diccionario de tomas** → estructura base con actores por toma.
* **Planificación y asistencia** → estructuras derivadas que permiten aplicar los algoritmos (fuerza bruta, Greedy, Tabú) y evaluar las soluciones.



In [6]:
import pandas as pd

# Cargar los datos desde el CSV
ruta = r"./datos_problema_doblaje.csv"
df = pd.read_csv(ruta, header=1) # Usamos header=1 para indicar que la segunda fila es el encabezado, donde aparecen que actores participan

# Eliminar la columna 'Unnamed: 11' y las filas que no son tomas reales
df = df.drop(columns=['Unnamed: 11'], errors='ignore')
df = df[pd.to_numeric(df['Toma'], errors='coerce').notna()]
df = df[df['Toma'] != 'TOTAL']

# Convertir 'Toma' a entero y establecerla como índice
df['Toma'] = df['Toma'].astype(int)
df = df.set_index('Toma')

print(df.head())

        1    2    3    4    5    6    7    8    9   10  Total
Toma                                                         
1     1.0  1.0  1.0  1.0  1.0  0.0  0.0  0.0  0.0  0.0    5.0
2     0.0  0.0  1.0  1.0  1.0  0.0  0.0  0.0  0.0  0.0    3.0
3     0.0  1.0  0.0  0.0  1.0  0.0  1.0  0.0  0.0  0.0    3.0
4     1.0  1.0  0.0  0.0  0.0  0.0  1.0  1.0  0.0  0.0    4.0
5     0.0  1.0  0.0  1.0  0.0  0.0  0.0  1.0  0.0  0.0    3.0


In [7]:
# Creamos un diccionario para saber qué actores participan en cada toma
tomas = {}

# Solo consideramos columnas que son números
actor_cols = [col for col in df.columns if str(col).isdigit()]
    
for toma_id, row in df.iterrows():
    # Recogemos los id de los actores para la toma
    actores_en_toma = {int(col) for col in actor_cols if row[col] == 1.0}
    tomas[toma_id] = actores_en_toma

# Comprobación de las primeras 5 tomas
for i, (toma_id, actores_set) in enumerate(tomas.items()):
    if i >= 5: 
        break
    print(f"Toma {toma_id}: actores {sorted(list(actores_set))}") 

Toma 1: actores [1, 2, 3, 4, 5]
Toma 2: actores [3, 4, 5]
Toma 3: actores [2, 5, 7]
Toma 4: actores [1, 2, 7, 8]
Toma 5: actores [2, 4, 8]


Según el modelo para el espacio de soluciones<br>

**4.(*)¿Cual es la función objetivo?**

Respuesta

La función objetivo consiste en minimizar el gasto total de los actores de doblaje.
 
 * Cada actor cobra por día asistido, independientemente del número de tomas que realice en ese día. 
 * Todos los actores cobran lo mismo
 
Por tanto, el coste total es proporcional a la suma de los días de asistencia de todos los actores.



In [8]:
from collections import defaultdict

def calcular_coste(planificacion, tomas_data):
    """
    planificacion: lista de días, cada día es lista de tomas
    tomas_data: dict[toma] -> set(actores)
    """
    asistencia = defaultdict(set)
    for dia_idx, dia in enumerate(planificacion, start=1):
        for toma in dia:  # <-- aquí cada 'dia' es una lista de tomas
            for actor in tomas_data[toma]:
                asistencia[actor].add(dia_idx)
    return sum(len(dias) for dias in asistencia.values())


**5.(*)¿Es un problema de maximización o minimización?**

Respuesta

Es un problema de **minimización**, ya que buscamos **minimizar el coste total** de la planificación.

El enunciado indica explícitamente que el objetivo es reducir al máximo el gasto por los servicios de los actores

**6.Diseña un algoritmo para resolver el problema por fuerza bruta**

Respuesta

El enfoque por fuerza bruta genera todas las permutaciones de las tomas y las agrupa en bloques de hasta 6, evaluando el coste de cada planificación.

Aunque conceptualmente correcto, este algoritmo es inviable para 30 tomas, ya que requiere **O(N!×N×A)** operaciones. Se puede aplicar solo en un problema reducido para comparar con heurísticas, en el ejemplo se han usado 9 tomas para apreciar mejor la diferencia con greedy

In [9]:
from itertools import permutations

# Fuerza bruta
def fuerza_bruta(tomas_data, max_tomas_dia=6):
    tomas_list = list(tomas_data.keys())
    mejor_coste = float('inf')
    mejor_plan = None
    for perm in permutations(tomas_list):
        # Agrupar en días según max_tomas_dia
        plan = [list(perm[i:i+max_tomas_dia]) for i in range(0, len(perm), max_tomas_dia)]
        coste = calcular_coste(plan, tomas_data)
        if coste < mejor_coste:
            mejor_coste = coste
            mejor_plan = plan
    return mejor_plan, mejor_coste



In [10]:
def greedy_planificacion(tomas, max_tomas_dia=6):
    """
    Construye una planificación tratando de minimizar la asistencia de actores:
    - Empieza con una toma libre.
    - Va añadiendo al día la toma que comparte más actores con las ya elegidas.
    - Repite hasta llenar el día (máx. 6 tomas).
    
    tomas: dict[int, set[int]] -> clave: toma_id, valor: actores de la toma
    """
    tomas_pendientes = set(tomas.keys())
    planificacion = []

    while tomas_pendientes:
        dia = []
        # elegimos una toma cualquiera para empezar el día
        actual = tomas_pendientes.pop()
        dia.append(actual)

        while len(dia) < max_tomas_dia and tomas_pendientes:
            # calculamos actores ya incluidos en el día
            actores_dia = set.union(*(tomas[t] for t in dia))
            # elegimos la toma pendiente que más solape tenga con los actores del día
            mejor_toma = max(
                tomas_pendientes,
                key=lambda t: len(actores_dia & tomas[t]),
            )
            tomas_pendientes.remove(mejor_toma)
            dia.append(mejor_toma)

        planificacion.append(dia)

    return planificacion


In [20]:
import random

# Elegimos 8 tomas aleatorias de entre las 30
tomas_reducidas = {k: tomas[k] for k in random.sample(list(tomas.keys()), 9)}

# Fuerza Bruta
plan_opt, coste_opt = fuerza_bruta(tomas_reducidas)
print("Fuerza bruta (óptimo):", plan_opt, "Coste:", coste_opt)


Fuerza bruta (óptimo): [[17, 8, 2, 22, 18, 11], [30, 25, 16]] Coste: 11


In [21]:
# Greedy sobre las mismas tomas
plan_greedy = greedy_planificacion(tomas_reducidas)
coste_greedy = calcular_coste(plan_greedy, tomas_reducidas)

print("Greedy:", plan_greedy, "Coste:", coste_greedy)

Greedy: [[2, 11, 22, 25, 8, 16], [17, 18, 30]] Coste: 12


#### Comparación de algoritmos sobre un subconjunto reducido

Para comprobar el rendimiento de los algoritmos, se ha seleccionado **un subconjunto de 9 tomas aleatorias** de las 30 totales.

| Algoritmo        | Coste | Tiempo de ejecución |
|-----------------|-------|------------------|
| Fuerza Bruta     | 11    | 4.7 s              |
| Greedy Mejorado  | 12    | 0 s              |

Observaciones:

- La **Fuerza Bruta** garantiza encontrar la planificación con coste mínimo (óptimo), pero el tiempo de cálculo crece factorialmente con el número de tomas.
- El **Greedy** construye la planificación rápidamente y aunque en este caso el coste es ligeramente superior (12 frente a 11), el tiempo de ejecución es prácticamente inmediato.
- Este ejemplo demuestra que para problemas de mayor tamaño, donde la fuerza bruta se vuelve inviable, las heurísticas como Greedy son necesarias para obtener soluciones aceptables en un tiempo razonable.


**7.Calcula la complejidad del algoritmo por fuerza bruta**

Respuesta

La complejidad del algoritmo por fuerza bruta es extremadamente alta debido a varios factores:

* **Factores de complejidad:**
    * **Permutaciones de tomas:** Se generan todas las posibles ordenaciones de las N tomas → O(N!)

    * **Agrupación en días:** Para cada permutación, se divide en grupos → O(N)

    * **Verificación de conflictos:** Para cada grupo, se verifica que no haya actores repetidos → O(A) donde A es el número máximo de actores por toma

* **Fórmula de complejidad:**

    * O(N!×N×A) = 2.65 x 10e32

Donde:

  * N = Número de tomas (30)
  * A = Número máximo de actores por toma (10)

In [13]:
import math

# Cálculo de la complejidad
N = 30
A = 10
complejidad = math.factorial(N) * N * A

print(f"Complejidad del algoritmo de fuerza bruta:")
print(f"O(N! × N × A) = O({N}! × {N} × {A})")
print(f"N! = {math.factorial(N):.3e}")
print(f"Operaciones totales: {complejidad:.3e}")
print(f"Este algoritmo es computacionalmente inviable para N=30")

Complejidad del algoritmo de fuerza bruta:
O(N! × N × A) = O(30! × 30 × 10)
N! = 2.653e+32
Operaciones totales: 7.958e+34
Este algoritmo es computacionalmente inviable para N=30


**8.(*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta**

Respuesta

#### Greedy
Un algoritmo Greedy (voraz) construye la planificación de manera secuencial:

* Recorre todas las tomas en el orden en que están dadas.
* Las agrupa en días de como máximo 6 tomas.
* No tiene en cuenta el solapamiento de actores entre tomas, simplemente reparte hasta llenar el límite de cada día.

**Ventajas:**

- Muy rápido (O(N² log N) como máximo).
- Proporciona una solución inicial válida en tiempo casi inmediato.

**Inconveniente:**

- La calidad de la solución puede ser baja, ya que no intenta optimizar la asistencia de actores ni reducir el coste total.
- Siempre devuelve la misma planificación y no explora alternativas.

#### Búsqueda Tabú
Para mejorar Greedy se utiliza Búsqueda Tabú:

* Se parte de una planificación inicial (puede ser Greedy o aleatoria).
* Se generan vecinos moviendo tomas de un día a otro o intercambiando tomas entre días, siempre respetando el máximo de 6 por día.
* Se evalúa el coste de cada vecino y se selecciona el mejor, incluso si no mejora la solución actual, siempre que no esté en la lista tabú.
* La lista tabú guarda movimientos recientes para evitar ciclos y forzar la exploración de nuevas soluciones.

**Ventajas:**

- Permite escapar de óptimos locales en los que Greedy queda atrapado.
- Encuentra sistemáticamente soluciones con menor coste que Greedy.
- Escala bien al problema completo (30 tomas), donde la fuerza bruta es inviable.

**Inconveniente:**

- No garantiza el óptimo global, aunque en la práctica obtiene costes mucho más bajos que Greedy.

In [14]:
import random
from collections import defaultdict
import copy

# ===============================
# Solución inicial aleatoria válida
# ===============================
def solucion_inicial_random(tomas, max_tomas_por_dia=6):
    tomas_list = list(tomas.keys())
    random.shuffle(tomas_list)
    planificacion = []
    for i in range(0, len(tomas_list), max_tomas_por_dia):
        planificacion.append(tomas_list[i:i+max_tomas_por_dia])
    return planificacion


# ===============================
# Generar vecinos
# ===============================
def generar_vecinos(plan, max_tomas_por_dia=6):
    vecinos = []
    n_dias = len(plan)

    # Movimiento 1: mover una toma de un día a otro
    for d1 in range(n_dias):
        for d2 in range(n_dias):
            if d1 != d2 and len(plan[d2]) < max_tomas_por_dia:
                for toma in plan[d1]:
                    nuevo_plan = copy.deepcopy(plan)
                    nuevo_plan[d1].remove(toma)
                    nuevo_plan[d2].append(toma)
                    if nuevo_plan[d1]:  # evitar días vacíos
                        vecinos.append(nuevo_plan)

    # Movimiento 2: intercambiar tomas entre dos días
    for d1 in range(n_dias):
        for d2 in range(d1+1, n_dias):
            for t1 in plan[d1]:
                for t2 in plan[d2]:
                    nuevo_plan = copy.deepcopy(plan)
                    nuevo_plan[d1].remove(t1)
                    nuevo_plan[d1].append(t2)
                    nuevo_plan[d2].remove(t2)
                    nuevo_plan[d2].append(t1)
                    vecinos.append(nuevo_plan)

    return vecinos


# ===============================
# Tabú Search
# ===============================
def tabu_search(tomas, max_tomas_por_dia=6, max_iter=200, tabu_tenure=10):
    plan_actual = solucion_inicial_random(tomas, max_tomas_por_dia)
    coste_actual = calcular_coste(plan_actual, tomas)
    mejor_plan, mejor_coste = plan_actual, coste_actual

    tabu_list = []

    for _ in range(max_iter):
        vecinos = generar_vecinos(plan_actual, max_tomas_por_dia)
        if not vecinos:
            break

        # Evaluar vecinos
        vecinos_eval = [(plan, calcular_coste(plan, tomas)) for plan in vecinos]

        # Ordenar por coste
        vecinos_eval.sort(key=lambda x: x[1])

        # Seleccionar el mejor no tabú
        for plan, coste in vecinos_eval:
            if plan not in tabu_list:
                plan_actual, coste_actual = plan, coste
                break

        # Actualizar mejor global
        if coste_actual < mejor_coste:
            mejor_plan, mejor_coste = plan_actual, coste_actual

        # Actualizar lista tabú
        tabu_list.append(plan_actual)
        if len(tabu_list) > tabu_tenure:
            tabu_list.pop(0)

    return mejor_plan, mejor_coste

In [15]:
# Greedy
plan_greedy = greedy_planificacion(tomas)
coste_greedy = calcular_coste(plan_greedy, tomas)

print("=== Planificación Greedy ===")
for i, dia in enumerate(plan_greedy, start=1):
    print(f"Día {i}: tomas {dia}")
print("Coste Greedy:", coste_greedy)

=== Planificación Greedy ===
Día 1: tomas [1, 6, 7, 11, 12, 20]
Día 2: tomas [2, 13, 22, 9, 25, 26]
Día 3: tomas [3, 4, 15, 5, 8, 10]
Día 4: tomas [14, 17, 18, 19, 23, 24]
Día 5: tomas [16, 27, 28, 29, 30, 21]
Coste Greedy: 31


In [16]:
# Tabú en 10 ejecuciones
costes_tabu = []
planes_tabu = []

for _ in range(10):
    plan, coste = tabu_search(tomas)
    costes_tabu.append(coste)
    planes_tabu.append((plan, coste))

mejor_plan_tabu, mejor_coste_tabu = min(planes_tabu, key=lambda x: x[1])

print("\n=== Comparativa Greedy vs Tabú ===")
print("Coste Greedy:", coste_greedy)
print("Tabú (10 ejecuciones):")
print("  Mejor coste:", min(costes_tabu))
print("  Peor coste: ", max(costes_tabu))
print("  Media:      ", sum(costes_tabu)/len(costes_tabu))

print("\n=== Mejor planificación Tabú ===")
for i, dia in enumerate(mejor_plan_tabu, start=1):
    print(f"Día {i}: tomas {dia}")
print("Coste Tabú (mejor):", mejor_coste_tabu)


=== Comparativa Greedy vs Tabú ===
Coste Greedy: 31
Tabú (10 ejecuciones):
  Mejor coste: 28
  Peor coste:  32
  Media:       29.5

=== Mejor planificación Tabú ===
Día 1: tomas [18, 23, 19, 24, 17, 14]
Día 2: tomas [6, 4, 3, 5, 7, 15]
Día 3: tomas [2, 29, 30, 26, 10, 8]
Día 4: tomas [20, 22, 21, 11, 12, 1]
Día 5: tomas [28, 27, 16, 9, 25, 13]
Coste Tabú (mejor): 28


#### Comparativa de algoritmos: Greedy Mejorado vs Búsqueda Tabú

Se ejecutaron los dos algoritmos sobre las 30 tomas del problema completo:

| Algoritmo        | Coste         | Tiempo aproximado |
|-----------------|---------------|-----------------|
| Greedy Mejorado  | 31            | 0.0 s           |
| Tabú (10 ejec.)  | 28 (mejor)    | 53.1 s          |

Observaciones:

1. **Greedy Mejorado**:
   - Construye la planificación rápidamente (tiempo prácticamente instantáneo).
   - Coste total: 31 días de asistencia de actores.
   - Limitación: es un algoritmo determinista y no explora otras soluciones posibles, por lo que no siempre alcanza el mínimo coste posible.

2. **Búsqueda Tabú**:
   - Se realizaron 10 ejecuciones independientes para explorar distintas soluciones.
   - Mejor coste obtenido: 28 días de asistencia de actores (3 días menos que Greedy).
   - Tiempo total: ~53 s para las 10 ejecuciones.
   - Ventaja: Tabú explora varias soluciones, refinando la planificación y mejorando sobre el resultado inicial de Greedy.

**Conclusión:**  
- Aunque **Greedy** es muy rápido y produce soluciones razonables, **Tabú** consigue reducir significativamente el coste total al explorar el espacio de soluciones.
- Esta comparativa demuestra que para problemas reales de tamaño completo, las heurísticas simples son un buen punto de partida, pero **metaheurísticas como Tabú son más efectivas** para minimizar el coste.


**9.(*)Calcula la complejidad del algoritmo**

Respuesta


**1. Greedy**
- Recorre todas las tomas y las ordena por número de actores no cubiertos.
- Itera hasta asignar todas las tomas.
- **Complejidad teórica:** O(N² log N)
- **Comentario:** Rápido, determinista, coste aceptable pero no garantiza óptimo.

**2. Tabú Search**
- Parte de Greedy, genera vecinos y mantiene lista tabú para evitar ciclos.
- Itera `max_iter` veces.
- **Complejidad teórica:** O(max_iter × N × A)
- **Comentario:** Explora más el espacio de soluciones y suele mejorar sobre Greedy y Local Search.

**Comparativa práctica**
- Fuerza Bruta: inviable para 30 tomas.
- Greedy: rápido, determinista.
- Tabú: mejor calidad de solución.


In [17]:
import math

# Parámetros del problema
N = 30   # número de tomas
A = 10   # número de actores
max_iter = 200  # número de iteraciones para Tabú

# Complejidad Greedy: O(N^2 log N)
complejidad_greedy = (N**2) * math.log(N, 2)

# Complejidad Tabú: O(max_iter × N × A)
complejidad_tabu = max_iter * N * A

print("=== Complejidad estimada de los algoritmos ===")
print(f"Greedy: O(N^2 log N) ≈ {complejidad_greedy:.2e} operaciones para N={N}")
print(f"Tabú:   O(max_iter × N × A) = {max_iter} × {N} × {A} = {complejidad_tabu:.2e} operaciones")


=== Complejidad estimada de los algoritmos ===
Greedy: O(N^2 log N) ≈ 4.42e+03 operaciones para N=30
Tabú:   O(max_iter × N × A) = 200 × 30 × 10 = 6.00e+04 operaciones


**10.Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios**

Respuesta

Se puede generar un dataset artificial con número configurable de tomas, actores y tamaño máximo por toma. Esto permite validar los algoritmos en escenarios de prueba más grandes o variados.

In [18]:
import pandas as pd
import random
from collections import defaultdict

def generar_datos_aleatorios(num_tomas=30, num_actores=10, max_actores_por_toma=5,):
    """
    Genera un DataFrame aleatorio con la misma estructura que el problema original.
    
    Args:
        num_tomas (int): Número de tomas a generar (default: 30)
        num_actores (int): Número total de actores (default: 10)
        max_actores_por_toma (int): Máximo de actores por toma (default: 5)
    
    Returns:
        pd.DataFrame: DataFrame con la misma estructura que el CSV original
        dict: Diccionario de tomas (para verificación)
    """
    
    # Generar datos aleatorios
    datos = []
    tomas = defaultdict(set)
    
    for toma_id in range(1, num_tomas + 1):
        # Seleccionar actores aleatorios para esta toma
        num_actores_toma = random.randint(1, max_actores_por_toma)
        actores_toma = random.sample(range(1, num_actores + 1), num_actores_toma)
        
        # Crear fila para el DataFrame
        fila = {'Toma': toma_id}
        for actor in range(1, num_actores + 1):
            fila[str(actor)] = 1.0 if actor in actores_toma else 0.0
        
        datos.append(fila)
        tomas[toma_id] = set(actores_toma)
    
    # Crear DataFrame
    columnas = ['Toma'] + [str(i) for i in range(1, num_actores + 1)] + ['Unnamed: 11']
    df = pd.DataFrame(datos, columns=columnas)
    
    # 3. Añadir fila de TOTALES (opcional)
    total_row = {'Toma': 'TOTAL'}
    for actor in range(1, num_actores + 1):
        total_row[str(actor)] = sum(1 for toma_actors in tomas.values() if actor in toma_actors)
    
    # Convertir total_row a un DataFrame de una sola fila para poder concatenarlo
    total_df = pd.DataFrame([total_row], columns=columnas)
    
    df = pd.concat([df, total_df], ignore_index=True)
    
    return df, tomas

# Ejemplo de uso
datos_aleatorios, tomas_aleatorias = generar_datos_aleatorios(
    num_tomas=30,
    num_actores=10,
    max_actores_por_toma=4,
)

# Mostrar primeras filas
print("Primeras 5 filas del DataFrame generado:")
print(datos_aleatorios.head())

# Verificar diccionario de tomas
print("\nEjemplo de tomas generadas:")
for toma_id in range(1, 6):
    print(f"Toma {toma_id}: Actores {sorted(tomas_aleatorias[toma_id])}")

Primeras 5 filas del DataFrame generado:
  Toma    1    2    3    4    5    6    7    8    9   10  Unnamed: 11
0    1  0.0  0.0  0.0  0.0  0.0  1.0  0.0  1.0  1.0  0.0          NaN
1    2  0.0  0.0  0.0  1.0  1.0  0.0  1.0  0.0  0.0  1.0          NaN
2    3  0.0  0.0  1.0  0.0  1.0  1.0  0.0  1.0  0.0  0.0          NaN
3    4  1.0  0.0  0.0  0.0  1.0  0.0  1.0  0.0  0.0  1.0          NaN
4    5  0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0  1.0  0.0          NaN

Ejemplo de tomas generadas:
Toma 1: Actores [6, 8, 9]
Toma 2: Actores [4, 5, 7, 10]
Toma 3: Actores [3, 5, 6, 8]
Toma 4: Actores [1, 5, 7, 10]
Toma 5: Actores [2, 9]


**11.Aplica el algoritmo al juego de datos generado**

Respuesta

In [19]:
# Greedy
plan_greedy_aleatorios = greedy_planificacion(tomas_aleatorias)
coste_greedy_aleatorios = calcular_coste(plan_greedy, tomas_aleatorias)

# Tabú en 10 ejecuciones
costes_tabu_aleatorios = []
planes_tabu_aleatorios = []

for _ in range(10):
    plan_aleatorios, coste_aleatorios = tabu_search(tomas_aleatorias)
    costes_tabu_aleatorios.append(coste_aleatorios)
    planes_tabu_aleatorios.append((plan_aleatorios, coste_aleatorios))

mejor_plan_tabu_aleatorios, mejor_coste_tabu_aleatorios = min(planes_tabu_aleatorios, key=lambda x: x[1])

print("=== Planificación Greedy ===")
for i, dia in enumerate(plan_greedy_aleatorios, start=1):
    print(f"Día {i}: tomas {dia}")
print("Coste Greedy:", coste_greedy_aleatorios)

print("\n=== Comparativa Greedy vs Tabú ===")
print("Coste Greedy:", coste_greedy_aleatorios)
print("Tabú (10 ejecuciones):")
print("  Mejor coste:", min(costes_tabu_aleatorios))
print("  Peor coste: ", max(costes_tabu_aleatorios))
print("  Media:      ", sum(costes_tabu_aleatorios)/len(costes_tabu_aleatorios))

print("\n=== Mejor planificación Tabú ===")
for i, dia in enumerate(mejor_plan_tabu_aleatorios, start=1):
    print(f"Día {i}: tomas {dia}")
print("Coste Tabú (mejor):", mejor_coste_tabu_aleatorios)

=== Planificación Greedy ===
Día 1: tomas [1, 22, 27, 2, 14, 3]
Día 2: tomas [4, 16, 12, 21, 6, 9]
Día 3: tomas [5, 15, 17, 24, 28, 8]
Día 4: tomas [7, 13, 29, 11, 10, 18]
Día 5: tomas [19, 20, 25, 26, 23, 30]
Coste Greedy: 44

=== Comparativa Greedy vs Tabú ===
Coste Greedy: 44
Tabú (10 ejecuciones):
  Mejor coste: 26
  Peor coste:  29
  Media:       27.7

=== Mejor planificación Tabú ===
Día 1: tomas [19, 5, 22, 1, 27, 28]
Día 2: tomas [16, 11, 4, 12, 29, 21]
Día 3: tomas [17, 15, 26, 25, 20, 30]
Día 4: tomas [8, 14, 2, 9, 6, 24]
Día 5: tomas [10, 23, 18, 7, 3, 13]
Coste Tabú (mejor): 26


#### Comparativa de algoritmos sobre datos aleatorios

Se generó un nuevo conjunto de datos aleatorios con 30 tomas y 10 actores. Los resultados de los algoritmos fueron los siguientes:

| Algoritmo        | Coste         | Tiempo aproximado |
|-----------------|---------------|-----------------|
| Greedy Mejorado  | 44            | ~0.0 s          |
| Tabú (10 ejec.)  | 26 (mejor)    | ~46.6 s           |

**Observaciones:**

1. **Greedy Mejorado**:
   - Construye la planificación rápidamente, en tiempo prácticamente instantáneo.
   - Coste total: 44 días de asistencia de actores.
   - Limitación: es determinista y no explora otras soluciones posibles, por lo que el coste puede ser mucho mayor que el óptimo.

2. **Búsqueda Tabú**:
   - Se realizaron 10 ejecuciones independientes para explorar distintas soluciones.
   - Mejor coste obtenido: 26 días de asistencia de actores.
   - Esto supone una **reducción de coste de casi la mitad** respecto a Greedy (44 → 26), demostrando la capacidad de Tabú para refinar la planificación inicial.
   - Tiempo total: ~44 s para las 10 ejecuciones, lo cual sigue siendo aceptable considerando la mejora significativa.

**Conclusión:**  
- En este caso, la diferencia de coste muestra claramente que **las soluciones Greedy pueden estar muy lejos del óptimo**, mientras que la **Búsqueda Tabú** explora el espacio de soluciones y consigue reducciones muy importantes.
- Esto confirma que para problemas reales con múltiples actores y tomas, las metaheurísticas son más efectivas para minimizar el coste total de asistencia de actores, mientras que Greedy puede servir solo como solución inicial rápida.


**12.Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo**

Respuesta

Para llevar a cabo el trabajo se han utilizado las siguientes referencias:

1. **Apuntes de la asignatura de Algoritmos de Optimización**, VIU 2025.  

2. **Apuntes de la asignatura de Python para la IA**, VIU 2025.  

3. **Documentación oficial de Python**  (collections, itertools, pandas)



**13.Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño**

Respuesta

El estudio del problema se podría avanzar considerando varias mejoras y variaciones:

1. **Costes diferentes por actor**:  
   - En la realidad, los protagonistas y actores secundarios cobran diferente.  
   - Se podría ponderar el coste de cada día por actor según su tarifa, en lugar de asumir un coste uniforme.

2. **Restricciones de secuencia de tomas**:  
   - Algunas tomas requieren que ciertos actores aparezcan juntos en días específicos (por ejemplo, cambios físicos o continuidad de vestuario).  
   - Esto introduce dependencias que podrían modelarse como restricciones adicionales en la planificación.

3. **Aumento del tamaño del problema**:  
   - Más actores, más tomas o más restricciones de disponibilidad podrían requerir algoritmos más avanzados.

4. **Optimización multiobjetivo**:  
   - Minimizar coste económico, minimizar días totales y balancear la carga de actores podría plantearse como un problema multiobjetivo.

En resumen, el problema actual es un modelo simplificado que permite explorar heurísticas, pero existen muchas extensiones realistas que lo acercan a un escenario profesional de doblaje.
